In [3]:
import numpy as np

raw_data = np.loadtxt('VTB.csv', delimiter=',', dtype=str, skiprows=1)
raw_data = np.char.strip(raw_data, '"')

open_p = raw_data[:, 2].astype(float)
min_p = raw_data[:, 4].astype(float)
max_p = raw_data[:, 3].astype(float)
close_p = raw_data[:, 1].astype(float)
prices = np.array([open_p, min_p, max_p, close_p]).T

mean_daily = np.mean(prices, axis=1).reshape(-1, 1)

merged_prices = np.hstack((prices, mean_daily))

min_val = np.amin(merged_prices, axis=0)
max_val = np.amax(merged_prices, axis=0)
mean_val = np.mean(merged_prices, axis=0)
variance = np.mean((merged_prices - mean_val) ** 2, axis=0)
std_val = np.std(merged_prices, axis=0)
rms_val = np.sqrt(variance)
cv_val = rms_val / mean_val


np.save('vtb_merged_prices.npy', merged_prices)

In [5]:
import pandas as pd

with open('VTB.csv', 'r', encoding='utf-8-sig') as f:
    lines = f.readlines()

rows = [line.strip().replace('"', '').split(',') for line in lines[1:10]]
rows[1][1] = '-'
rows[2][1] = ' '
rows[2][2] = 'н/д'
rows[4][2] = '-'

with open('VTB_dirty.csv', 'w', encoding='utf-8') as f:
    f.write('Date,Price,Open,Vol.,Change %\n')
    for r in rows:
        vol_clean = r[5].replace('M', '').replace('K', '')
        f.write(f"{r[0]},{r[1]},{r[2]},{vol_clean},{r[6]}\n")

df = pd.read_csv(
    'VTB_dirty.csv',
    header=0,
    usecols=['Date', 'Open', 'Price', 'Vol.'],
    parse_dates=['Date'],
    decimal='.',
    na_values=['-', ' ', 'н/д'],
    dtype={'Open': 'float64', 'Price': 'float64', 'Vol.': 'float64'}
)

display(df)
display(df.dtypes.to_frame(name='Тип данных'))

display(df.sort_values(by='Vol.', ascending=False))

df_ranked = df.copy()
df_ranked['Ранг_Open'] = df['Open'].rank(ascending=False)
display(df_ranked[['Date', 'Open', 'Ранг_Open']])

display(df[['Open', 'Price']].describe())

display(pd.DataFrame({
    'Метрика': ['Корреляция', 'Ковариация'],
    'Значение': [df['Price'].corr(df['Vol.']), df['Price'].cov(df['Vol.'])]
}))

df_dropped = df.dropna(thresh=3)
display(df_dropped)

df_filled = df_dropped.copy()
price_means = df_filled[['Open', 'Price']].mean(axis=1)
df_filled['Open'] = df_filled['Open'].fillna(price_means)
df_filled['Price'] = df_filled['Price'].fillna(price_means)
df_filled['Vol.'] = df_filled['Vol.'].fillna(df_filled['Vol.'].mean())
display(df_filled)

df_past = pd.DataFrame({
    'Date': pd.to_datetime(['2025-08-01', '2025-08-02']),
    'Open': [48.5, 48.9],
    'Price': [48.7, 49.1],
    'Vol.': [55.0, 60.0]
})
display(pd.concat([df_filled, df_past], ignore_index=True))
display(pd.merge(df_filled, df_past, on='Date', how='outer'))

df_overlap = pd.DataFrame({
    'Date': [df_filled['Date'].iloc[0], pd.to_datetime('2026-09-01')],
    'Open': [df_filled['Open'].iloc[0], 52.0],
    'Price': [df_filled['Price'].iloc[0], 52.5],
    'Vol.': [df_filled['Vol.'].iloc[0], 80.0]
})
display(pd.concat([df_filled, df_overlap], ignore_index=True).drop_duplicates(subset=['Date'], keep='last'))
display(pd.merge(df_filled, df_overlap, on=['Date', 'Open', 'Price', 'Vol.'], how='outer'))

df_other_stock = pd.DataFrame({
    'Date': df_filled['Date'],
    'Open_SBER': [270.0] * len(df_filled),
    'Price_SBER': [272.5] * len(df_filled),
    'Vol_SBER': [150.0] * len(df_filled)
})
display(pd.concat([df_filled, df_other_stock.drop(columns=['Date'])], axis=1))
display(pd.merge(df_filled, df_other_stock, on='Date', how='inner'))

,Date,Price,Open,Vol.
0,2026-08-31,51.87,50.30,73.37
1,2026-08-30,NaN,50.30,7.99
2,2026-08-29,NaN,NaN,4.27
3,2026-08-28,50.16,50.80,60.33
4,2026-08-27,50.79,NaN,64.83
5,2026-08-26,50.08,51.29,66.49
6,2026-08-25,51.44,51.26,92.50
7,2026-08-24,51.17,52.75,97.65
8,2026-08-23,52.10,52.75,18.15


,Тип данных
Date,datetime64[us]
Price,float64
Open,float64
Vol.,float64


,Date,Price,Open,Vol.
7,2026-08-24,51.17,52.75,97.65
6,2026-08-25,51.44,51.26,92.50
0,2026-08-31,51.87,50.30,73.37
5,2026-08-26,50.08,51.29,66.49
4,2026-08-27,50.79,NaN,64.83
3,2026-08-28,50.16,50.80,60.33
8,2026-08-23,52.10,52.75,18.15
1,2026-08-30,NaN,50.30,7.99
2,2026-08-29,NaN,NaN,4.27


,Date,Open,Ранг_Open
0,2026-08-31,50.30,6.5
1,2026-08-30,50.30,6.5
2,2026-08-29,NaN,NaN
3,2026-08-28,50.80,5.0
4,2026-08-27,NaN,NaN
5,2026-08-26,51.29,3.0
6,2026-08-25,51.26,4.0
7,2026-08-24,52.75,1.5
8,2026-08-23,52.75,1.5


,Open,Price
count,7.000000,7.000000
mean,51.350000,51.087143
std,1.035954,0.788706
min,50.300000,50.080000
25%,50.550000,50.475000
50%,51.260000,51.170000
75%,52.020000,51.655000
max,52.750000,52.100000


...

,Date,Price_x,Open_x,Vol._x,Open_y,Price_y,Vol._y
0,2025-08-01,NaN,NaN,NaN,48.5,48.7,55.0
1,2025-08-02,NaN,NaN,NaN,48.9,49.1,60.0
2,2026-08-23,52.10,52.75,18.15,NaN,NaN,NaN
3,2026-08-24,51.17,52.75,97.65,NaN,NaN,NaN
4,2026-08-25,51.44,51.26,92.50,NaN,NaN,NaN
5,2026-08-26,50.08,51.29,66.49,NaN,NaN,NaN
6,2026-08-27,50.79,50.79,64.83,NaN,NaN,NaN
7,2026-08-28,50.16,50.80,60.33,NaN,NaN,NaN
8,2026-08-30,50.30,50.30,7.99,NaN,NaN,NaN
9,2026-08-31,51.87,50.30,73.37,NaN,NaN,NaN


,Date,Price,Open,Vol.
1,2026-08-30,50.30,50.30,7.99
2,2026-08-28,50.16,50.80,60.33
3,2026-08-27,50.79,50.79,64.83
4,2026-08-26,50.08,51.29,66.49
5,2026-08-25,51.44,51.26,92.50
6,2026-08-24,51.17,52.75,97.65
7,2026-08-23,52.10,52.75,18.15
8,2026-08-31,51.87,50.30,73.37
9,2026-09-01,52.50,52.00,80.00


,Date,Price,Open,Vol.
0,2026-08-23,52.10,52.75,18.15
1,2026-08-24,51.17,52.75,97.65
2,2026-08-25,51.44,51.26,92.50
3,2026-08-26,50.08,51.29,66.49
4,2026-08-27,50.79,50.79,64.83
5,2026-08-28,50.16,50.80,60.33
6,2026-08-30,50.30,50.30,7.99
7,2026-08-31,51.87,50.30,73.37
8,2026-09-01,52.50,52.00,80.00


,Date,Price,Open,Vol.,Open_SBER,Price_SBER,Vol_SBER
0,2026-08-31,51.87,50.30,73.37,270.0,272.5,150.0
1,2026-08-30,50.30,50.30,7.99,270.0,272.5,150.0
3,2026-08-28,50.16,50.80,60.33,270.0,272.5,150.0
4,2026-08-27,50.79,50.79,64.83,270.0,272.5,150.0
5,2026-08-26,50.08,51.29,66.49,270.0,272.5,150.0
6,2026-08-25,51.44,51.26,92.50,270.0,272.5,150.0
7,2026-08-24,51.17,52.75,97.65,270.0,272.5,150.0
8,2026-08-23,52.10,52.75,18.15,270.0,272.5,150.0


,Date,Price,Open,Vol.,Open_SBER,Price_SBER,Vol_SBER
0,2026-08-31,51.87,50.30,73.37,270.0,272.5,150.0
1,2026-08-30,50.30,50.30,7.99,270.0,272.5,150.0
2,2026-08-28,50.16,50.80,60.33,270.0,272.5,150.0
3,2026-08-27,50.79,50.79,64.83,270.0,272.5,150.0
4,2026-08-26,50.08,51.29,66.49,270.0,272.5,150.0
5,2026-08-25,51.44,51.26,92.50,270.0,272.5,150.0
6,2026-08-24,51.17,52.75,97.65,270.0,272.5,150.0
7,2026-08-23,52.10,52.75,18.15,270.0,272.5,150.0
